<a href="https://colab.research.google.com/github/charre2021/polars_work_on_cyber_attacks/blob/main/Cyber_Attacks_Financial_And_Market_Impact.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [63]:
import polars as pl
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import NumeralTickFormatter
import polars.selectors as cs
import kagglehub
import requests
import os

In [ ]:
path = kagglehub.dataset_download("aryanmdev/cyber-attacks-financial-and-market-impact")

100%|██████████| 169k/169k [00:00<00:00, 42.2MB/s]

Extracting files...


In [ ]:
files_list = os.listdir(path)
market_impact_path = os.path.join(path, files_list[0])
financial_impact_path = os.path.join(path, files_list[1])
incidents_master_path = os.path.join(path, files_list[2])

In [ ]:
market_impact_raw = pl.read_csv(market_impact_path)

In [ ]:
financial_impact_raw = pl.read_csv(financial_impact_path)

In [ ]:
incidents_master_raw = pl.read_csv(incidents_master_path)

In [ ]:
financial_impact_raw.filter(pl.col("notes") != "null").select(pl.col("notes"))

notes
str
"""Insurance claim partially deni…"
"""Loss methodology: Monte Carlo …"
"""Customer notification costs in…"
"""Insurance claim partially deni…"
"""Regulatory fine imposed under …"
…
"""Recovery costs include $16.6M …"
"""Costs allocated across FY2024 …"
"""Loss methodology: Monte Carlo …"


In [ ]:
financial_impact_raw.filter(pl.col("created_at") != pl.col("created_at").first()).select(pl.col("created_at"))

created_at
str


In [ ]:
financial_impact_raw.filter(pl.col("updated_at") != pl.col("updated_at").first()).select(pl.col("updated_at"))

updated_at
str


In [ ]:
fi_remove_last_two_cols = financial_impact_raw.drop(["created_at","updated_at"])

In [ ]:
fi_remove_last_two_cols.head()

incident_id,direct_loss_usd,direct_loss_method,ransom_demanded_usd,ransom_paid_usd,ransom_source,recovery_cost_usd,legal_fees_usd,regulatory_fine_usd,insurance_payout_usd,total_loss_usd,total_loss_method,total_loss_lower_bound,total_loss_upper_bound,inflation_adjusted_usd,cpi_index_used,notes
str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,str,f64,f64,f64,str,str
"""2021-0508-001""",1.26e7,"""disclosed""",1.3803e7,null,null,9.4554e6,2.4965e6,90695.25,6.7563e6,2.4643e7,"""calculated""",1.5348e7,4.3747e7,2.9238e7,"""CPI-U 2021 (270.97)""",null
"""2025-1211-001""",7.6405e6,"""disclosed""",null,null,null,5.8572e6,1.8092e6,null,2.6910e6,1.5307e7,"""disclosed""",1.0206e7,1.8906e7,1.5307e7,"""CPI-U 2025 (321.5)""",null
"""2023-0115-001""",3.4882e7,"""calculated""",null,null,null,2.6404e7,1.0331e7,null,3.1760e7,7.1616e7,"""disclosed""",6.0854e7,1.0515e8,7.5565e7,"""CPI-U 2023 (304.702)""",null
"""2021-0315-001""",4.6822e6,"""disclosed""",null,null,null,3.6429e6,1.0290e6,null,1.7725e6,9354133.8,"""disclosed""",7.6490e6,1.4525e7,1.1098e7,"""CPI-U 2021 (270.97)""",null
"""2021-1204-001""",2.6846e6,"""estimated""",null,null,null,2.5749e6,206822.23,null,null,5.4663e6,"""estimated""",3.5198e6,6.7558e6,6.4856e6,"""CPI-U 2021 (270.97)""",null


In [ ]:
CPI_U_2026 = 326.588

fi_add_dates_and_cpis = fi_remove_last_two_cols.with_columns(
    (pl.col("incident_id").str.extract(r"(\d{4}-\d{4})", 1).str.to_date("%Y-%m%d").alias("incident_date")),
    (pl.col("cpi_index_used").str.extract(r"\s{1}(\d{4})", 1).cast(pl.Int32).alias("cpi_base_year")),
    (pl.col("cpi_index_used").str.extract(r"\((\d+\.\d+)\)", 1).cast(pl.Float64).alias("base_cpi")),
    (pl.lit(CPI_U_2026).alias("cpi_u_2026"))
    )

In [ ]:
fi_add_dates_and_cpis_revised = fi_add_dates_and_cpis.drop(["inflation_adjusted_usd", "cpi_index_used"])

In [ ]:
revised_calculations = fi_add_dates_and_cpis_revised.select(["recovery_cost_usd", "legal_fees_usd", "regulatory_fine_usd", "insurance_payout_usd", "total_loss_usd"]).fill_null(0)

In [ ]:
revised_calculations.with_columns(
    (
        revised_calculations.select(["recovery_cost_usd", "legal_fees_usd", "regulatory_fine_usd"]).sum_horizontal() - pl.col("insurance_payout_usd")
        ).alias("new_total")
).drop(["recovery_cost_usd", "legal_fees_usd", "regulatory_fine_usd", "insurance_payout_usd"])

In [ ]:
fi_add_dates_and_cpis_revised["direct_loss_method"].value_counts()

direct_loss_method,count
str,u32
"""estimated""",358
"""disclosed""",220
"""calculated""",200


In [ ]:
incidents_master_raw.select(["incident_id", "company_name"])\
.join(fi_add_dates_and_cpis_revised.select("incident_id"), on = "incident_id", how = "full")\
.filter(pl.col("incident_id_right").is_null())

incident_id,company_name,incident_id_right
str,str,str
"""2023-1119-001""","""Canberra Sciences Pty Ltd.""",null
"""2023-0214-001""","""Delhi BioSciences Holdings Ltd…",null
"""2025-0315-001""","""PortAero Logistics Ltd.""",null
"""2023-0914-001""","""WattAmp Power Group Inc.""",null
"""2025-0826-001""","""Phoenix Capital Corp.""",null
…,…,…
"""2021-0821-001""","""CreditSecura Group Holdings In…",null
"""2023-0227-002""","""Jensen Medical Holdings Inc.""",null
"""2023-0714-002""","""Sterling Forge Markets Holding…",null


In [49]:
bounds_graph_data = fi_add_dates_and_cpis_revised.with_columns(
    ((pl.col("total_loss_usd") * (pl.col("cpi_u_2026") / pl.col("base_cpi"))).alias("cpi_adjusted_total_loss")),
    ((pl.col("total_loss_lower_bound") * (pl.col("cpi_u_2026") / pl.col("base_cpi"))).alias("cpi_adjusted_total_loss_lower_bound")),
    ((pl.col("total_loss_upper_bound") * (pl.col("cpi_u_2026") / pl.col("base_cpi"))).alias("cpi_adjusted_total_loss_upper_bound")),
    ((pl.col("insurance_payout_usd") * (pl.col("cpi_u_2026") / pl.col("base_cpi"))).alias("cpi_adjusted_insurance_payout")),
).select(["incident_date", "cpi_adjusted_total_loss", "cpi_adjusted_total_loss_lower_bound", "cpi_adjusted_total_loss_upper_bound", "cpi_adjusted_insurance_payout"])\
.sort("incident_date")\
.fill_null(0)

In [67]:
output_notebook()
p = figure(title = "Cyber Attack Loss Amounts", x_axis_type = "datetime", width = 1200)
x_for_all = bounds_graph_data["incident_date"]

p.line(x = x_for_all, y = bounds_graph_data["cpi_adjusted_total_loss"])
p.yaxis[0].formatter = NumeralTickFormatter(format = "$0,0")
show(p)
